In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

os.environ["SPARK_LOCAL_IP"] = "localhost"

conf = (
    SparkConf()
    .setMaster("local[*]")
    .setAppName("Delta Lake Spark Jupyter Lab")
    .set("spark.driver.memory", "30g")
    .set("spark.executor.memory", "30g")
    .set("spark.driver.cores", "16")
    .set("spark.worker.cores", "16")
    .set("spark.jars", "/home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/*")
    .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .set(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .set(
        "spark.sql.delta.catalogImpl", "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)
sc = SparkContext(conf=conf).getOrCreate()
spark: SparkSession = SparkSession(sc)

24/06/29 09:20:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/29 09:20:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/06/29 09:20:28 ERROR SparkContext: Failed to add /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* to Spark environment
java.io.FileNotFoundException: Jar /home/baptvit/Documents/github/mineracao-dados-massivos/apps/jars/* not found
	at org.apache.spark.SparkContext.addLocalJarFile$1(SparkContext.scala:2100)
	at org.apache.spark.SparkContext.addJar(SparkContext.scala:2156)
	at org.apache.spark.SparkContext.$anonfun$new$15(SparkContext.scala:526)
	at org.apache.spark.SparkContext.$anonfun$new$15$adapted(SparkContext.scala:526)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62

In [2]:
spark

In [3]:
df = spark.read.parquet("/home/baptvit/Documents/github/mineracao-dados-massivos/data/med-qa-dataset/medqa_kb")

In [4]:
df.printSchema()

root
 |-- sentence: string (nullable = true)
 |-- embedding_sentence: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- token_sentence: integer (nullable = true)
 |-- metadata: string (nullable = true)



In [5]:
df.show()

+-------------------------------------+--------------------+--------------+--------------------+
|                             sentence|  embedding_sentence|token_sentence|            metadata|
+-------------------------------------+--------------------+--------------+--------------------+
|成、高脂血症、口服避孕药、缺血性脑...|[-0.1020278483629...|            27|Raw file name is:...|
|              消化腺alim entary gl...|[0.03708112984895...|            32|Raw file name is:...|
|             饵摄入量与血压呈负相关。|[-0.1035428270697...|            14|Raw file name is:...|
|                 Available at: www...|[-0.1713246852159...|            38|Raw file name is:...|
|                 TS is the only co...|[-0.1271661520004...|           110|Raw file name is:...|
|                 Hochler H, Yafe H...|[-0.2039959281682...|            50|Raw file name is:...|
|                 Thyrotropin-relea...|[0.00182249711360...|            28|Raw file name is:...|
|   对HBV感染育龄期及妊娠母亲的管理...|[0.19435840845108...|            92|R

# Evaluation para Textbook

In [ ]:
PD_KB.count()

In [ ]:
import pandas as pd
pd_textbook_test = pd.read_csv("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_textbook_test_embedding_1000.csv")

In [ ]:
pd_textbook_test.head()

In [ ]:
pd_textbook_test.count()

In [ ]:
import time
import ast
import numpy as np

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances


def euclidean_distances_f(query_vector, df, vector_col="embedding_sentence", top_n=10):
    """
    Finds the top N most similar vectors to a query vector in a pandas DataFrame,
    using Euclidean distance.

    Args:
        df (pd.DataFrame): The input DataFrame containing the vectors.
        vector_col (str): The name of the column containing the vectors.
        query_vector (list): The query vector to compare against.
        top_n (int, optional): The number of top most similar vectors to return.
            Defaults to 10.

    Returns:
        pd.DataFrame: A new DataFrame containing the top N most similar vectors
            and their corresponding distances.
    """

    # Ensure query vector has the same length as columns in the vector column
    # if len(query_vector) != len(df[vector_col].iloc[0]):
    #     raise ValueError("Query vector length must match the number of features in the vector column.")
    
    start_time = time.time()
    # Efficiently calculate Euclidean distances using sklearn.metrics.pairwise.euclidean_distances

    query_vector= [np.array(ast.literal_eval(query_vector[1:-1]), dtype="float64")]
    distances = euclidean_distances(query_vector, df[vector_col].to_numpy())

    # Combine vectors with their distances into a DataFrame
    df_with_distances = pd.concat([df[vector_col], pd.DataFrame(distances, columns=['distance'])], axis=1)

    # Sort by distance (ascending order) and select the top N most similar vectors
    top_similar = df_with_distances.sort_values(by='distance').head(top_n)

    return top_similar, (time.time() - start_time)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def cosine_similarity_f(df, vector_col, query_vector, top_n=10, exclude_self=True):
    """
    Finds the top N most similar vectors in a pandas DataFrame to a given query vector
    using cosine similarity.

    Args:
        df (pandas.DataFrame): The DataFrame containing the vectors.
        vector_col (str): The name of the column containing the vector representations.
        query_vector (list): The query vector to compare against.
        top_n (int, optional): The number of most similar vectors to return. Defaults to 10.
        exclude_self (bool, optional): Whether to exclude the query vector itself from
                                       the results. Defaults to True.

    Returns:
        pandas.DataFrame: A new DataFrame containing the top N most similar vectors
                          and their corresponding cosine similarity scores.
    """
    start_time = time.time()
    # Ensure query vector is a NumPy array for efficient calculations
    query_vector = np.array(query_vector)

    # Extract vectors from the DataFrame
    vectors = df[vector_col].to_numpy()

    # Calculate cosine similarities efficiently using vectorization
    similarities = cosine_similarity(query_vector.reshape(1, -1), vectors)[:, 0]

    # Sort indices by similarity in descending order
    sorted_indices = similarities.argsort()[::-1]

    if exclude_self:
        # Exclude the query vector if specified
        sorted_indices = sorted_indices[1:top_n + 1]

    # Create a new DataFrame with top N similar vectors and their similarities
    top_similar_df = pd.DataFrame({
        'index': df.index[sorted_indices],
        'similarity': similarities[sorted_indices],
        vector_col: df[vector_col].iloc[sorted_indices]
    })

    return top_similar_df, (time.time() - start_time)

In [ ]:
import pandas as pd
from numpy import abs
from scipy.spatial.distance import pdist

def manhattan_distance_f(df, vector_col, query_vector, top_n=10):
  """
  Finds the top N most similar vectors to a query vector in a pandas DataFrame,
  using Manhattan distance.

  Args:
      df (pandas.DataFrame): The input DataFrame containing the vectors.
      vector_col (str): The name of the column containing the vectors.
      query_vector (list-like): The query vector to compare against.
      top_n (int, optional): The number of top similar vectors to return. Defaults to 10.

  Returns:
      pandas.DataFrame: A new DataFrame containing the top N most similar vectors
          and their distances to the query vector.
  """

  # Ensure consistent vector shapes
  start_time = time.time()
  query_vector = pd.Series(query_vector)
  df[vector_col] = df[vector_col].apply(pd.Series)  # Convert to Series for consistent shape

  # Calculate pairwise Manhattan distances using pdist for efficiency
  distances = pdist(df[vector_col].tolist(), metric='cityblock')  # 'cityblock' for Manhattan

  # Create a temporary DataFrame with distances
  temp_df = pd.DataFrame({'index': df.index, 'distance': distances.flatten()})

  # Sort by distance (ascending) and select top N (excluding query vector itself)
  temp_df = temp_df.sort_values(by='distance')
  top_similar = temp_df.iloc[1:top_n+1, :]  # Exclude row 0 (query vector)
  top_similar = df.iloc[top_similar['index'].tolist()]

  return top_similar, (time.time() - start_time)


In [ ]:
def search_f(rewrited_sentence_embedding, algoritimo_name, kb_dataset=PD_KB, top_k=10):
    if algoritimo_name == "euclidean_distance":
        pd_top_10, time = euclidean_distances_f(rewrited_sentence_embedding, kb_dataset, top_k)
    elif algoritimo_name == "cosine_similarity":
        pd_top_10, time = cosine_similarity_f(rewrited_sentence_embedding, kb_dataset, top_k)
    elif algoritimo_name == "manhattan_distance":
        pd_top_10, time = manhattan_distance_f(rewrited_sentence_embedding, kb_dataset, top_k)
        
    return pd_top_10, time

In [ ]:
import datetime 

def evaluate(pd_test, algoritimo_name):
    algoritimos_name = []
    times = []
    timestamps = []
    top_1_result = []  # If true, the model is write on top 1 else false
    top_5_result = (
        []
    )  # If true, the model is write on top 5 results else false
    top_10_result = (
        []
    )  # If true, the model is write on top 10 results else false
    distances = []
    query_id_result = []
    for rows in pd_test.iterrows():
        breakpoint()
        row = rows[1] 
        query_id = row[0]
        expected_output = row["sentence"]
        #input_rewrited_sentence = row["rewrited_sentence"]
        rewrited_sentence_embedding = row["rewrited_sentence_embedding"]

        # level of diferent types of testing (vector as input and expect value match)
        timestamps.append(datetime.datetime.now())

        ### Create a search function
        pd_search_results, time = search_f(rewrited_sentence_embedding, algoritimo_name, kb_dataset=PD_KB, top_k=10)
        ###
        
        times.append(time)

        algoritimos_name.append(algoritimo_name)
        retrived_distance = 1
        for num, rows in enumerate(pd_search_results.iterrows()):  # top 10 results
            row = rows[1]
            # If is first hit
            top_1 = False
            top_5 = False
            top_10 = False
            retrived_sentence = row["sentence"]
            retrived_distance = row["distance"]
            if retrived_sentence == expected_output and num == 0:
                top_1 = True
                top_5 = True
                top_10 = True
                distance = retrived_distance
                break
            elif retrived_sentence == expected_output and num <= 5:
                top_1 = False
                top_5 = True
                top_10 = True
                distance = retrived_distance
                break
            elif retrived_sentence == expected_output and num <= 10:
                top_1 = False
                top_5 = False
                top_10 = True
                distance = retrived_distance
                break
            else:
                top_1 = False
                top_5 = False
                top_10 = False
                distance = 1

        distances.append(distance)

        top_1_result.append(top_1)
        top_5_result.append(top_5)
        top_10_result.append(top_10)
        query_id_result.append(query_id)

        msg = f"Query id {query_id},Top 1: {top_1}, Within Top 5 {top_5}, Within Top 10 {top_10} Query time: {time}s, Distance {retrived_distance} and with"
        print(msg)
    
    metrics = {
        "timestamp": timestamps,
        "algoritimos_name": algoritimos_name,
        "query_id": query_id_result,
        "search_duration": times,
        "top1": top_1_result,
        "top5": top_5_result,
        "top10": top_10_result,
        "distance": distances,
    }
    df = pd.DataFrame(metrics)
    df.to_csv(f"/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/evaluation/{algoritimos_name}.csv")

In [ ]:
pd_textbook_test.head(1)["rewrited_sentence_embedding"].values

In [ ]:
numpy_array_asarray = np.asarray(PD_KB['embedding_sentence'].tolist())

In [ ]:
PD_KB["teste"] = PD_KB['embedding_sentence'].tolist()

In [ ]:
PD_KB.dtypes

In [18]:
query_vector = '[-0.08195801079273224, 0.1827828586101532, 0.1738424301147461, -0.29751360416412354, 0.43829524517059326, 0.0701017677783966, 0.49329015612602234, 0.5026525855064392, 0.016669658944010735, -0.12393417209386826, -0.12785358726978302, -0.32508233189582825, -0.24144278466701508, 0.3633136451244354, -9.764337301021442e-05, 0.6132171750068665, 0.4952540695667267, 0.020313631743192673, -0.2811177372932434, 0.2601775825023651, -0.042864445596933365, -0.07395979017019272, -0.13984213769435883, 0.30202993750572205, 0.3438684940338135, -0.059475235641002655, -0.028720734640955925, 0.10006221383810043, -0.19929932057857513, -0.4337463080883026, 0.5151315927505493, 0.14401564002037048, -0.06938505917787552, -0.24099206924438477, 0.15052048861980438, 0.16505102813243866, -0.11976425349712372, -0.2950705289840698, -0.23436225950717926, -0.06727396696805954, -0.5717906951904297, -0.39379483461380005, -0.20047840476036072, 0.07477575540542603, 0.07076775282621384, -0.0413530133664608, 0.4155977964401245, -0.0238066166639328, -0.03894141688942909, 0.22270254790782928, -0.30552420020103455, 0.16222068667411804, -0.009017270058393478, -0.04234757646918297, 0.3730045557022095, 0.4958335757255554, -0.37217244505882263, -0.5104745626449585, -0.3045472204685211, -0.3355746567249298, 0.3463534116744995, 0.053330548107624054, 0.1978514939546585, -0.4479193091392517, 0.2394714206457138, 0.1873447746038437, -0.08832996338605881, 0.5287754535675049, -0.5356414914131165, 0.0758529007434845, -0.4724203050136566, -0.012738167308270931, -0.08964473009109497, -0.07365590333938599, -0.19285611808300018, 0.2174725979566574, -0.02826687879860401, -0.10242385417222977, -0.03516340255737305, -0.029164928942918777, -0.31846603751182556, 0.2937155067920685, -0.20349480211734772, 0.3669956922531128, 0.11247102171182632, 0.09585917741060257, 0.01453172042965889, 0.17461879551410675, 0.04719046875834465, 0.33316391706466675, 0.028317248448729515, 0.21822050213813782, 0.1402418464422226, 0.03619939461350441, -0.05810996890068054, -0.03575830161571503, 0.23257479071617126, 0.18291276693344116, -0.1636464148759842, -0.007872369140386581, 0.4053744971752167, -0.18237946927547455, -0.03220990672707558, 0.02755051851272583, -0.10661792010068893, 0.04343079403042793, 0.3561233580112457, 0.06489862501621246, -0.2878096103668213, 0.171388640999794, 0.09965284913778305, 0.07445493340492249, -0.09694760292768478, -0.419158935546875, -0.33585476875305176, 0.19075770676136017, 0.10690956562757492, 0.11132468283176422, -0.06787289679050446, 0.20398832857608795, -0.07692385464906693, -0.3748852014541626, 0.009676715359091759, 0.47009721398353577, -0.4017202854156494, 0.1249300017952919, 0.13528017699718475, 0.22663092613220215, -0.06675336509943008, -0.018561800941824913, 0.12059123814105988, -0.01233524177223444, 0.18734581768512726, -0.2800258696079254, -0.43857818841934204, 0.41478386521339417, -0.23127789795398712, -0.04572150483727455, -0.15376047790050507, 0.12592455744743347, -0.036960747092962265, -0.512503981590271, 0.21594120562076569, -0.10527699440717697, 0.14592590928077698, -0.18415316939353943, 0.2221389263868332, 0.18973839282989502, -0.016188081353902817, 0.5971071124076843, 0.21004779636859894, 0.17355188727378845, -0.02245550975203514, -0.15072141587734222, -0.20719550549983978, -0.10162895917892456, -0.2868208587169647, 0.03456885740160942, -0.28492510318756104, 0.08509165793657303, 0.5104677677154541, 0.03882625326514244, -0.3940410912036896, 0.44078052043914795, 0.014112458564341068, 0.19916968047618866, -0.11504814773797989, 0.38295701146125793, -0.06314515322446823, 0.352386474609375, -0.009100271388888359, -0.2323630303144455, 0.615742564201355, -0.02081352472305298, 0.1944899708032608, 0.03798431158065796, 0.20082077383995056, 0.07569774240255356, 0.2760165333747864, -0.12545563280582428, -1.1551446914672852, 0.34974080324172974, 0.2273150235414505, 0.35851407051086426, 0.052263181656599045, -0.4099876582622528, -0.04137487709522247, -0.5094351768493652, -0.22710196673870087, -0.32974037528038025, -0.06730721890926361, -0.4183444678783417, -0.398192822933197, -0.0676298588514328, 0.12981268763542175, -0.42065003514289856, -0.05785815417766571, 0.07935798913240433, -0.5599684715270996, 0.038375373929739, 0.18247580528259277, 0.06414355337619781, 0.0856688991189003, 0.2226337492465973, -0.29457151889801025, -0.3200784921646118, 0.06623572111129761, -0.27168285846710205, -0.1412004977464676, 0.23195531964302063, -0.18875183165073395, 0.23337364196777344, -0.13265185058116913, -0.08393661677837372, 0.25392988324165344, 0.1999192237854004, -0.15133579075336456, 0.2927089035511017, 0.06921471655368805, -0.021835539489984512, 0.041377052664756775, 0.025504304096102715, -0.4507753551006317, 0.25658565759658813, -0.15424887835979462, 0.7001768350601196, 0.04347608610987663, -0.41944050788879395, 0.23871886730194092, 0.4008105397224426, 0.027439912781119347, -0.0509856641292572, 0.697527289390564, 0.01750996522605419, -0.13105997443199158, 0.13590525090694427, -0.14209482073783875, -0.2344081699848175, 0.07265853881835938, -0.5325650572776794, -0.38324683904647827, 0.4935702681541443, 0.25892165303230286, 0.049955401569604874, 0.3336910605430603, -0.1329440176486969, -0.009537318721413612, 0.48134565353393555, -0.4543246924877167, 0.08651407808065414, -0.3535354733467102, -0.17614677548408508, 0.07499951124191284, -0.2989279329776764, 0.07221263647079468, -0.44436153769493103, -0.12339656800031662, -0.33228152990341187, -0.0031224777922034264, -0.020648710429668427, 0.08319578319787979, 0.24780558049678802, 0.2859719395637512, 0.059441905468702316, -0.5230242609977722, -0.5094887614250183, 0.19178003072738647, 0.1688181757926941, 0.04161873459815979, 0.04252292215824127, 0.15019339323043823, 0.05589548125863075, -0.4756081998348236, 0.4531986713409424, -0.47118431329727173, -0.20318342745304108, 0.42154285311698914, 0.1417415291070938, -0.019726699218153954, -0.0880650207400322, 0.10132403671741486, 0.34016066789627075, -0.23675145208835602, 0.1364476978778839, -0.0438033826649189, -0.20441296696662903, 0.0667666420340538, 0.08866111189126968, -0.149020716547966, -0.3593484163284302, -0.29228293895721436, 0.355743944644928, -0.22095909714698792, -0.2229287326335907, 0.4161251187324524, 0.12641173601150513, 0.21758143603801727, 0.3668913245201111, 0.4026907980442047, -0.2523845434188843, -0.0904822126030922, -0.4012766480445862, 0.10578006505966187, 0.34372904896736145, -0.3705988824367523, 0.13598594069480896, 0.03498820587992668, -0.48430177569389343, -3.2444965839385986, 0.02020963840186596, 0.09598327428102493, -0.28124862909317017, 0.2719656527042389, 0.054798368364572525, 0.19395358860492706, -0.052443377673625946, -0.39779967069625854, 0.33959466218948364, -0.010659999214112759, -0.29673537611961365, 0.5199912786483765, 0.13649024069309235, 0.06268603354692459, -0.1588994264602661, 0.034087710082530975, -0.253390908241272, -0.03487252816557884, 0.4707913100719452, -0.1856406182050705, -0.42500731348991394, -0.05331554263830185, -0.020957978442311287, 0.24220392107963562, 0.25247320532798767, -0.1182025596499443, -0.186279758810997, -0.40643805265426636, -0.33684659004211426, 0.1245613843202591, -0.2714378535747528, 0.0024240976199507713, 0.36510688066482544, 0.0036916069220751524, 0.3924740254878998, -0.10832851380109787, -0.2710970342159271, -0.06922592967748642, -0.17509131133556366, -0.2716212570667267, -0.6957812905311584, -0.09378398954868317, 0.1450732946395874, 0.7303535342216492, 0.004077656660228968, 0.2171015739440918, -0.1979154795408249, 0.24123191833496094, 0.24849964678287506, 0.0769997090101242, -0.013547006994485855, -0.13200746476650238, 0.1076139286160469, -0.09023737907409668, 0.058799512684345245, 0.5688296556472778, 0.3769133687019348, -0.48195964097976685, -0.3485089838504791, -0.04444395750761032, -0.24073563516139984, -0.6686665415763855, -0.19813011586666107, -0.16031503677368164, -0.32228440046310425, -0.6672295928001404, -0.17768298089504242, 0.12612976133823395, 0.17094828188419342, 0.223081573843956, 0.529056966304779, -0.24493268132209778, -0.46590739488601685, -0.14294318854808807, -0.5413497686386108, 0.1854075938463211, 0.020655209198594093, -0.08995363861322403, -0.22413960099220276, -0.16499067842960358, -0.21419194340705872, -0.06731529533863068, 0.09113290160894394, 0.2600807249546051, -0.07753169536590576, 0.3555927276611328, 0.025498943403363228, -0.40265917778015137, -0.5468896627426147, 0.2898479700088501, 0.3346443772315979, -0.010046835988759995, 0.20368507504463196, -0.21140052378177643, 0.13158762454986572, 0.2908490300178528, -0.03424907103180885, 0.613216757774353, -0.1872139424085617, -0.11496227979660034, -0.1344071626663208, 0.6963808536529541, -0.001671121339313686, -0.07343299686908722, 0.24314163625240326, -0.317395955324173, -0.15578630566596985, 0.2425706386566162, -0.16745908558368683, 0.2794049382209778, -0.44062989950180054, 0.3963615894317627, -0.37451013922691345, 0.12422138452529907, -0.08733997493982315, 0.1348782628774643, 0.617599368095398, -0.023883294314146042, 0.10620003193616867, 0.10830295085906982, 0.48562267422676086, -0.4011434018611908, -0.2941398322582245, -0.22686585783958435, -0.0902930423617363, 0.0009537264122627676, 0.2389073669910431, 0.004663330502808094, -0.011473463848233223, -0.20034576952457428, -0.05249243602156639, -0.14582011103630066, 0.24786944687366486, 0.08799342066049576, 0.013853774406015873, 0.03597182407975197, -0.22323231399059296, -0.032980095595121384, 0.10057084262371063, 0.09772074967622757, 0.11306937783956528, -0.2405814230442047, -0.17157535254955292, -0.170368492603302, 0.1602213829755783, 0.009081616066396236, 0.010614633560180664, -0.08177115768194199, 0.07391047477722168, -0.2207106500864029, -0.106255903840065, -0.0946325957775116, -0.19772036373615265, -0.04090157151222229, 0.35178494453430176, 0.16037511825561523, -0.1602345108985901, -0.017190970480442047, -0.7098275423049927, 0.07353512197732925, 0.026343461126089096, 0.1518182009458542, -0.24285395443439484, -0.2740679681301117, 0.5193015933036804, 0.009014740586280823, -0.03020692616701126, -0.09080641716718674, 0.1399671584367752, -0.06791382282972336, -0.19321659207344055, 0.27944454550743103, -0.6127201914787292, -0.04299606382846832, 0.4599728584289551, -0.11855887621641159, -0.08595633506774902, -0.16936726868152618, 0.19675476849079132, 0.05566757172346115, -0.3581247329711914, -0.4157339930534363, -0.13120563328266144, 0.25958016514778137, 0.03342530503869057, 0.228769451379776, -0.23849701881408691, -0.053154781460762024, -0.06429897248744965, -0.08605200797319412, 0.04496303200721741, -0.084450863301754, -0.14890046417713165, -0.24102680385112762, -0.14336951076984406, 0.1342540830373764, -0.2620716392993927, 0.10702890902757645, 0.42107462882995605, 0.38255542516708374, -0.2221994549036026, -0.3594277501106262, -0.17010165750980377, 0.09606657177209854, -0.31482449173927307, 0.08520658314228058, -0.010320700705051422, -0.4502441883087158, 0.2523066997528076, -0.46542537212371826, -0.2507402300834656, -0.4219890534877777, -0.05340287461876869, 0.05524608492851257, -0.262439101934433, -0.16508720815181732, -0.2899884879589081, -0.2335769236087799, 0.17742644250392914, 0.04606836289167404, 0.11920630931854248, 0.16842785477638245, 0.09687957912683487, -0.37366047501564026, -0.25265762209892273, 0.23802553117275238, -0.027984460815787315, -0.023461680859327316, -0.0549757182598114, -0.24666959047317505, -0.607570230960846, -0.0389457568526268, -0.22458812594413757, 0.07056407630443573, -0.14978572726249695, 0.18869321048259735, -0.2537253499031067, 0.18389533460140228, -0.007803992368280888, -0.06795987486839294, 0.262797474861145, -0.1140718013048172, -0.20407715439796448, 0.2639974057674408, -0.1602758914232254, -0.13055361807346344, 0.18437319993972778, -0.22748695313930511, 0.3097470998764038, -0.08268297463655472, -0.04310714080929756, -0.24277910590171814, -0.3492477238178253, 0.0393880233168602, -0.4293530285358429, -0.23461613059043884, -0.10041598975658417, 0.007808315567672253, 0.21356038749217987, 0.1481468379497528, -0.17449621856212616, 0.004184792283922434, 0.1224886104464531, 0.20399083197116852, 0.010896483436226845, 0.06853046268224716, 0.5847233533859253, -0.031879737973213196, -0.38829943537712097, 0.16123831272125244, -0.3967111110687256, 0.13485844433307648, -0.21179552376270294, 0.06995700299739838, 0.14942428469657898, 0.279585599899292, -0.37564921379089355, 0.2857205867767334, -0.44356799125671387, -0.25271251797676086, 0.12455742806196213, 0.1518831104040146, -0.14719456434249878, 0.19039711356163025, -0.4846446216106415, -0.06345777213573456, 0.2252851277589798, -0.3588131070137024, -0.17017164826393127, 0.3752439320087433, -0.0689798891544342, -0.13202837109565735, 0.13357986509799957, -0.1574336588382721, 0.19457392394542694, 0.46132034063339233, 0.47139379382133484, 0.17556539177894592, 0.10522379726171494, -0.19348590075969696, 0.34946712851524353, 0.15785393118858337, -0.09785302728414536, -0.1806403249502182, 0.014794982969760895, -0.028819236904382706, -0.8972061276435852, -0.048037026077508926, 0.13665705919265747, -0.036048635840415955, -0.3615776002407074, 0.7128046154975891, 0.4137531518936157, -0.4561513662338257, -0.14496882259845734, -0.02732381597161293, 0.03935784474015236, -0.02760648913681507, 0.052549514919519424, 0.1134478822350502, 0.17083176970481873, 0.7093715667724609, 0.22729790210723877, 0.27787813544273376, 0.3051452040672302, -0.005595289636403322, 0.02035619132220745, -0.06183111295104027, 0.4232396185398102, 0.09390975534915924, 0.0891072154045105, 0.18296882510185242, 0.49972447752952576, -0.055082134902477264, -0.06050392612814903, -0.04953327402472496, -0.060302458703517914, -0.19387328624725342, 0.05779479444026947, 0.09042318165302277, 0.3176855742931366, 0.3785550892353058, 0.48740673065185547, 0.28724542260169983, 0.2725469768047333, 0.034608568996191025, 0.08706119656562805, 0.5349887609481812, -0.09052343666553497, 0.43483811616897583, -0.175481915473938, 0.26733511686325073, 0.20358403027057648, -0.00858466885983944, 0.37476444244384766, 0.45654651522636414, 0.09369180351495743, -0.2639586329460144, 0.39513111114501953, 0.03962525725364685, 0.008306551724672318, 0.3601735830307007, -0.49350792169570923, -0.03580981120467186, -0.3355981111526489, 0.08501693606376648, -0.4781306982040405, -0.011640111915767193, 0.0745762437582016, 0.18073156476020813, 0.02066619321703911, -0.5450002551078796, -0.1897585391998291, -0.0067848749458789825, 0.4349685609340668, 0.018441446125507355, -0.03243041783571243, -0.13178101181983948, -0.4776872992515564, -0.35446280241012573, -0.29600027203559875, -0.5104959607124329, -0.40733498334884644, -0.4335746765136719, -0.21888606250286102, 0.14974704384803772, -0.006224704906344414, 0.012142164632678032, -0.15799660980701447, -0.035949692130088806, 0.21715307235717773, 0.1596554070711136, -0.23642191290855408, -0.22123777866363525, -0.0923672690987587, 0.5089373588562012, -0.08918318897485733, 0.5304841995239258, -0.42551615834236145, -0.007630255073308945, -0.11935266852378845, -0.34521248936653137, -0.16598740220069885, 0.18255481123924255, 0.13483811914920807, 0.13883329927921295, 0.4850500822067261, -0.5063029527664185, -0.18790018558502197, 0.0047674234956502914, 0.3115975260734558, -0.6072988510131836, 0.29612046480178833, 0.5189732909202576, -0.03330704942345619, 0.028875011950731277, -0.14111636579036713, -0.20734632015228271, -0.07056192308664322, -0.12341564893722534, -0.4273393452167511, -0.04136054217815399, 0.2674737572669983, -0.7492296099662781, 0.11326780170202255, 0.49409326910972595, 0.3073815405368805, 0.2083798050880432, -0.16221016645431519, -0.24666427075862885, 0.2186339646577835, -0.05304941162467003, 0.06607355177402496, 0.010708263143897057, -0.5344719886779785, 0.0187959223985672, 0.06395558267831802, -0.2750823497772217, 0.06942091882228851, -0.028762659057974815, 0.040757983922958374, 0.24939800798892975, 0.025126632302999496, 0.3142337501049042, 0.048675812780857086, -0.17876432836055756, 0.01946331560611725, 0.02439475618302822, -0.39668235182762146, -0.1278488039970398, -0.0818590596318245, 0.08317483216524124, -0.23987741768360138, -0.1504906564950943, -0.33262404799461365, -0.04431597888469696, -0.3055804371833801, -0.25070029497146606, 0.2242109328508377]'

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances


def euclidean_distances_f(query_vector, df, vector_col="embedding_sentence", top_n=10):
    """
    Finds the top N most similar vectors to a query vector in a pandas DataFrame,
    using Euclidean distance.

    Args:
        df (pd.DataFrame): The input DataFrame containing the vectors.
        vector_col (str): The name of the column containing the vectors.
        query_vector (list): The query vector to compare against.
        top_n (int, optional): The number of top most similar vectors to return.
            Defaults to 10.

    Returns:
        pd.DataFrame: A new DataFrame containing the top N most similar vectors
            and their corresponding distances.
    """

    # Ensure query vector has the same length as columns in the vector column
    # if len(query_vector) != len(df[vector_col].iloc[0]):
    #     raise ValueError("Query vector length must match the number of features in the vector column.")
    
    start_time = time.time()
    # Efficiently calculate Euclidean distances using sklearn.metrics.pairwise.euclidean_distances

    query_vector= [np.array(ast.literal_eval(query_vector[1:-1]), dtype="float64")]
    distances = euclidean_distances(query_vector, df[vector_col])

    # Combine vectors with their distances into a DataFrame
    df_with_distances = pd.concat([df[vector_col], pd.DataFrame(distances, columns=['distance'])], axis=1)

    # Sort by distance (ascending order) and select the top N most similar vectors
    top_similar = df_with_distances.sort_values(by='distance').head(top_n)

    return top_similar, (time.time() - start_time)

euclidean_distances_f(query_vector, df=PD_KB, vector_col="teste")

In [43]:
from pyspark.sql.types import FloatType
from pyspark.sql.functions import col, udf, lit
from pyspark.ml.linalg import Vectors, VectorUDT

def euclidean_distance(v1, v2):
    """Calculates the Euclidean distance between two vectors."""
    import math
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(v1, v2)))

euclidean_distance_udf = udf(euclidean_distance, FloatType())


In [ ]:
df.printSchema()

In [ ]:
distances = df.withColumn("distance", euclidean_distance_udf(col("embedding_sentence"), [np.array(ast.literal_eval(query_vector[1:-1]), dtype="float64")]))  # Replace with your reference vector


In [ ]:
distances.show()

In [40]:
data = spark.createDataFrame([("test_id", query_vector)], ["id", "query_vector"])

In [41]:

# Apply the UDF to convert the string vector
data = data.withColumn("query_vector_array", convert_udf(col("query_vector")))

data.show()


+-------+--------------------+--------------------+
|     id|        query_vector|  query_vector_array|
+-------+--------------------+--------------------+
|test_id|[-0.0819580107927...|[-0.0819580107927...|
+-------+--------------------+--------------------+



In [42]:
data.printSchema()

root
 |-- id: string (nullable = true)
 |-- query_vector: string (nullable = true)
 |-- query_vector_array: array (nullable = true)
 |    |-- element: double (containsNull = true)



In [46]:
distances = data.withColumn("distance", data(col("query_vector_array"), lit(list(ast.literal_eval(query_vector[1:-1])))))  # Replace with your reference vector


TypeError: 'DataFrame' object is not callable

In [48]:
distances = data.withColumn("distance", euclidean_distance_udf(col("query_vector_array"), lit(list(ast.literal_eval(query_vector[1:-1])))))  # Replace with your reference vector


In [49]:
distances.show()

+-------+--------------------+--------------------+--------+
|     id|        query_vector|  query_vector_array|distance|
+-------+--------------------+--------------------+--------+
|test_id|[-0.0819580107927...|[-0.0819580107927...|     0.0|
+-------+--------------------+--------------------+--------+



In [14]:
df_question_test = spark.read.load("/home/baptvit/Documents/github/mineracao-dados-massivos/evaluation/evaluation/data/pd_question_test_embedding_refat_1000")